# Introduction to AI: E-commerce Conversion Prediction
This notebook complements the lecture slides in `slides/introduction_to_ai/slides.md`.

## Story
An online shop wants to understand **customer segments** (unsupervised learning) and **predict conversions** (supervised learning).
We will use a **simple synthetic e-commerce dataset** with intuitive features.

**Columns**
- `customer_id`
- `age`, `gender`
- `income_k`, `sessions_last30`, `avg_basket`, `time_on_site_min`
- `utm_source`, `device`, `region`, `ui_variant`
- `converted` (label: 1 = purchase, 0 = no purchase)

**Goal:** predict whether a new customer will convert and discuss which UI variant performs best.

## Exercises
- Exercise 1: Explore the dataset and define features/label.
- Exercise 2: Cluster customers with k-means and interpret segments.
- Exercise 3: Train a classifier to predict `converted` and evaluate it.

Notes:
- Data generation and train/test split are already provided.
- `predict_df` is the unlabeled data for your predictions.
- Work through the notebook from top to bottom.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, ConfusionMatrixDisplay

In [ ]:
# Synthetic e-commerce data (already prepared for you)
rng = np.random.default_rng(42)
n = 1200

df_raw = pd.DataFrame(
    {
        "customer_id": np.arange(1, n + 1),
        "age": rng.integers(18, 70, size=n),
        "gender": rng.choice(["female", "male"], size=n, p=[0.52, 0.48]),
        "income_k": np.round(rng.normal(60, 20, size=n).clip(20, 150), 1),
        "sessions_last30": rng.poisson(3, size=n).clip(0, 20),
        "avg_basket": np.round(rng.normal(80, 40, size=n).clip(10, 300), 1),
        "time_on_site_min": np.round(rng.normal(6, 3, size=n).clip(1, 30), 1),
        "utm_source": rng.choice(
            ["organic", "paid", "social", "email", "referral"],
            size=n,
            p=[0.35, 0.25, 0.2, 0.1, 0.1],
        ),
        "device": rng.choice(["mobile", "desktop", "tablet"], size=n, p=[0.6, 0.3, 0.1]),
        "region": rng.choice(["EU", "US", "APAC"], size=n, p=[0.4, 0.35, 0.25]),
        "ui_variant": rng.choice(["A", "B"], size=n, p=[0.5, 0.5]),
    }
)

# Create conversion probability (simple, interpretable signal)
logit = -3.0
logit += 0.03 * (df_raw["age"] - 35)
logit += 0.04 * (df_raw["income_k"] - 50)
logit += 0.15 * df_raw["sessions_last30"]
logit += 0.02 * (df_raw["avg_basket"] - 60)
logit += 0.10 * (df_raw["time_on_site_min"] - 5)
logit += np.where(df_raw["utm_source"] == "email", 0.8, 0.0)
logit += np.where(df_raw["utm_source"] == "paid", 0.3, 0.0)
logit += np.where(df_raw["utm_source"] == "social", -0.2, 0.0)
logit += np.where(df_raw["device"] == "desktop", 0.2, 0.0)
logit += np.where(df_raw["ui_variant"] == "B", 0.25, 0.0)
logit += rng.normal(0, 0.8, size=n)

prob = 1 / (1 + np.exp(-logit))
df_raw["converted"] = rng.binomial(1, prob)

train_df, test_df = train_test_split(
    df_raw,
    test_size=0.25,
    random_state=42,
    stratify=df_raw["converted"],
)

predict_df = test_df.drop(columns=["converted"]).copy()
y_test = test_df["converted"].copy()


def score_predictions(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    print(f"Accuracy: {acc:.3f}")
    return acc


df_raw.head()

In [ ]:
# Exercise 1: Explore and prepare the data
# Tasks:
# 1) Inspect df_raw (shape, columns, missing values).
# 2) Define numeric_cols and categorical_cols.
# 3) Create feature_cols = numeric_cols + categorical_cols.
# 4) Set cluster_features = numeric_cols (for k-means).
# 5) Create X_train, y_train, X_test, y_test using train_df/test_df.
#
# Hint:
# numeric_cols = ["age", "income_k", "sessions_last30", "avg_basket", "time_on_site_min"]
# categorical_cols = ["gender", "utm_source", "device", "region", "ui_variant"]
#
# Later you will one-hot encode categorical columns with pd.get_dummies().

df = df_raw.copy()

# Your code here

## Part A: Unsupervised learning - k-means clustering
We ignore the label and group customers purely by behavior. This mimics a real **segmentation** use case.

### Tasks
1. Scale the features.
2. Fit k-means with a chosen k (start with k=4).
3. Inspect cluster sizes and interpret clusters using feature averages.

In [ ]:
# Exercise 2: K-means clustering
# 1) Use cluster_features from Exercise 1
# 2) Scale them
# 3) Fit KMeans and get cluster labels
# 4) Add clusters to a copy of df
# 5) Inspect cluster sizes and feature averages per cluster

# Your code here

In [ ]:
# Visualize clusters in 2D with PCA
# 1) Fit PCA on X_scaled and project to 2D
# 2) Create a scatter plot colored by cluster
# 3) Label axes and add a title

# Your code here

## Part B: Supervised learning - classification
Now we use the label `converted` and train a model to **predict** whether a customer converts.
Train/test sets are already defined for you.

### Tasks
1. One-hot encode categorical features.
2. Build a pipeline with scaling + a classifier.
3. Train, predict, and evaluate.

In [ ]:
# Exercise 3: Classification
# 1) One-hot encode categorical columns (pd.get_dummies)
# 2) Align train and test columns
# 3) Train a classifier
# 4) Predict on X_test and score with score_predictions(y_test, y_pred)
# 5) (Optional) Predict on predict_df for the competition

# Your code here

In [ ]:
# Plot the confusion matrix for your classifier
# Hint: ConfusionMatrixDisplay.from_estimator(...)

# Your code here

## Discussion and extensions
- Try different values of k and justify your choice.
- Compare clusters to `converted` and discuss mismatches.
- Swap the classifier (DecisionTree, RandomForest, SVM) and compare metrics.
- Compare performance with and without feature scaling.
- Which UI variant performs better overall? Does it depend on device or source?